<a href="https://colab.research.google.com/github/LujainAlawwad/ImplicitAspectExtraction/blob/main/4_ArabicModel_Inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Installations:

### Unsloth installations:

In [ ]:
# %%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install trl==0.22.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 49.2 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.8/110.8 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import bitsandbytes as bnb
import unsloth
from unsloth import FastLanguageModel

import transformers, accelerate, peft

print("Unsloth loaded successfully ✓")
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("peft:", peft.__version__)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth loaded successfully ✓
transformers: 4.56.2
accelerate: 1.13.0
peft: 0.19.1


In [ ]:
import torch, torchvision
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)


torch: 2.10.0+cu128
torchvision: 0.25.0+cu128


In [ ]:
!pip install transformers==4.56.2

### installation #2:


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install ipython-autotime -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 51.0 MB/s eta 0:00:00


In [ ]:
load_ext autotime

time: 215 µs (started: 2026-05-03 11:18:00 +00:00)


In [ ]:
from IPython.display import Markdown, HTML
import numpy as np

time: 474 µs (started: 2026-05-03 11:18:00 +00:00)


In [ ]:
from google.colab import userdata
# openai_key=userdata.get('openAi_key')
myGPT = userdata.get('MyGPT')
hf_key=userdata.get('hf_key')

time: 1.32 s (started: 2026-05-03 11:18:00 +00:00)



## Inference:

### Load the model.

In [ ]:
!nvidia-smi


Sun May  3 11:18:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P0             29W /   70W |     105MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from unsloth import FastLanguageModel
from peft import PeftModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

BASE =  "unsloth/llama-3-8b-bnb-4bit"          # <-- use the *actual* base you fine-tuned
ADAPTER = "LujainAbdulrahman/finetuned_Arabic_se16_correct"  # your adapter repo
ADAPTER = "LujainAbdulrahman/finetuned_Arabic_m-absa_3epochs"
# ADAPTER = "LujainAbdulrahman/finetuned_Arabic_haad_10epochs"


# 1) load base
base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = BASE,
    max_seq_length  = max_seq_length,
    dtype           = dtype,
    load_in_4bit    = load_in_4bit,
)

# 2) attach LoRA adapter
model = PeftModel.from_pretrained(base_model, ADAPTER)

# 3) switch to inference fast path (optional)
FastLanguageModel.for_inference(model)


==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096, padding_idx=128255)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=128, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=128, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
      

time: 33.5 s (started: 2026-05-03 11:18:31 +00:00)


### CAMeL tools for Arabic prosessing:

In [ ]:
    # from google.colab import drive
    # drive.mount('/content/drive')

    # # # Create a directory for packages in your Drive if it doesn't exist
    # # !mkdir -p /content/drive/My Drive/colab_packages

    # # # Install packages to this location
    # # !pip install camel-tools -t "/content/drive/My Drive/colab_packages"
    # !pip install unsloth unsloth_zoo xformers -t "/content/drive/My Drive/colab_packages"

    # # Add the package directory to your Python path
# import sys
# sys.path.append('/content/drive/My Drive/colab_packages')

time: 453 µs (started: 2026-05-03 11:25:12 +00:00)


In [ ]:
!pip install -q camel-tools
%pip install -q --upgrade --force-reinstall camel-tools==1.5.0
import camel_tools
print("camel_tools version:", camel_tools.__version__)


ERROR: Ignored the following yanked versions: 0.2.dev0, 0.3.dev0, 0.4.dev0, 0.4.dev1, 0.4.dev2, 0.4.dev3, 0.4.dev4, 0.4.dev5, 1.0.0
ERROR: Ignored the following versions that require a different python version: 1.5.0 Requires-Python >=3.7.0, <3.11; 1.5.1 Requires-Python >=3.7.0, <3.11; 1.5.2 Requires-Python >=3.7.0, <3.11; 1.5.3 Requires-Python <3.12,>=3.8.0; 1.5.4 Requires-Python <3.12,>=3.8.0
ERROR: Could not find a version that satisfies the requirement camel-tools==1.5.0 (from versions: 1.0.1, 1.1.0, 1.2.0, 1.3.0, 1.3.1, 1.4.0, 1.4.1, 1.5.5, 1.5.6, 1.5.7)
ERROR: No matching distribution found for camel-tools==1.5.0
camel_tools version: 1.5.7
time: 7.51 s (started: 2026-05-03 11:25:12 +00:00)


In [ ]:
# ===== CAMeL bootstrap: install+link morphology DB + MLE disambiguator, then smoke-load =====
import os, pathlib, subprocess

HOME = pathlib.Path.home()
ROOT = HOME / ".camel_tools" / "data"

def _install(pkg_id: str):
    try:
        print(f"Installing CAMeL data package: {pkg_id} ...")
        subprocess.run(["camel_data", "-i", pkg_id], check=True)
        print("  ✓ done")
    except FileNotFoundError:
        raise RuntimeError(
            "camel_data command not found. Install camel-tools first:\n"
            "  pip install camel-tools\n"
            "Then re-run this cell."
        )

def _ensure_file(package_dir: pathlib.Path, alt_dirname: str, filename: str, installer_id: str):
    """
    Ensure `package_dir/<dirname>/filename` exists. If alt layout exists, symlink.
    """
    # Two possible layouts (dirnames):
    calima_dir = package_dir / "calima-msa-r13"
    msa_dir    = package_dir / "msa-r13"

    calima_file = calima_dir / filename
    msa_file    = msa_dir / filename

    # If neither exists, try to install
    if not calima_file.exists() and not msa_file.exists():
        _install(installer_id)

    # Recheck after install
    has_calima = calima_file.exists()
    has_msa    = msa_file.exists()

    # If still neither, try to locate anywhere under package_dir
    if not has_calima and not has_msa and package_dir.exists():
        for p in package_dir.rglob(filename):
            # Create links to both names pointing to the first found file
            target = p
            calima_dir.mkdir(parents=True, exist_ok=True)
            msa_dir.mkdir(parents=True, exist_ok=True)
            try:
                if not calima_file.exists():
                    calima_file.symlink_to(os.path.relpath(target, calima_file.parent))
                if not msa_file.exists():
                    msa_file.symlink_to(os.path.relpath(target, msa_file.parent))
            except Exception as e:
                print("  (symlink skipped)", e)
            has_calima = calima_file.exists()
            has_msa    = msa_file.exists()
            break

    # If exactly one exists, link the other to it
    if has_calima and not has_msa:
        msa_dir.mkdir(parents=True, exist_ok=True)
        try:
            if not msa_file.exists():
                msa_file.symlink_to(os.path.relpath(calima_file, msa_file.parent))
        except Exception as e:
            print("  (symlink skipped)", e)
    elif has_msa and not has_calima:
        calima_dir.mkdir(parents=True, exist_ok=True)
        try:
            if not calima_file.exists():
                calima_file.symlink_to(os.path.relpath(msa_file, calima_file.parent))
        except Exception as e:
            print("  (symlink skipped)", e)

    # Final check
    if not calima_file.exists():
        raise FileNotFoundError(f"Expected file not found after install: {calima_file}")

    return calima_file  # return the canonical calima-msa-r13 path

# 1) Ensure morphology DB
morph_dir = ROOT / "morphology_db"
morph_file = _ensure_file(
    package_dir=morph_dir,
    alt_dirname="msa-r13",
    filename="morphology.db",
    installer_id="morphology-db-msa-r13",
)
print("✅ Morphology DB:", morph_file)

# 2) Ensure MLE disambiguator model
mle_dir = ROOT / "disambig_mle"
mle_file = _ensure_file(
    package_dir=mle_dir,
    alt_dirname="msa-r13",
    filename="model.json",
    installer_id="disambig-mle-calima-msa-r13",
)
print("✅ MLE model:", mle_file)

# 3) Smoke-load both to verify
from camel_tools.morphology.database import MorphologyDB
from camel_tools.morphology.analyzer import Analyzer
from camel_tools.disambig.mle import MLEDisambiguator

db = MorphologyDB(str(morph_file))
analyzer = Analyzer(db)  # Analyzer.builtin_analyzer('calima-msa-r13') would also work now
mle = MLEDisambiguator.pretrained()  # now resolves to ~/.camel_tools/data/disambig_mle/calima-msa-r13/model.json

print("✅ Loaded MorphologyDB, Analyzer, and MLEDisambiguator.pretrained(). Ready to use.")


✅ Morphology DB: /root/.camel_tools/data/morphology_db/calima-msa-r13/morphology.db
✅ MLE model: /root/.camel_tools/data/disambig_mle/calima-msa-r13/model.json
✅ Loaded MorphologyDB, Analyzer, and MLEDisambiguator.pretrained(). Ready to use.
time: 4.82 s (started: 2026-05-03 11:25:19 +00:00)


###Helper functions:

In [ ]:
def normalize_and_dedup_list(terms):
    """Normalize each term and de-duplicate by normalized key.
    Keeps the first surface form as the representative."""
    if terms is None:
        return []
    items = terms if isinstance(terms, list) else parse_aspect_string(terms)
    seen, out = set(), []
    for w in items:
        key = normalize_phrase(w)
        if key and key not in seen:
            seen.add(key)
            out.append(w)  # keep original surface for readability
    return out


time: 718 µs (started: 2026-05-03 11:25:24 +00:00)


In [ ]:
import ast
import pandas as pd
import numpy as np

def clean_paraphrases(x):
    if isinstance(x, str):
        try:
            x = ast.literal_eval(x)
        except Exception:
            return [x]  # fallback: keep as single string

    # Sometimes it's a list of one big string that itself is a list
    while isinstance(x, list) and len(x) == 1 and isinstance(x[0], str):
        try:
            x = ast.literal_eval(x[0])
        except Exception:
            break

    # Finally, make sure it's a flat list of strings
    if isinstance(x, list) and all(isinstance(i, str) for i in x):
        return x
    return [str(x)]


# Convert string representation of lists to actual lists
# test['paraphrases'] = test['paraphrases'].apply(clean_paraphrases)
# test['aspect'] = test['aspect'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
#split  test['paraphrases']  based on \n :
# test['paraphrases'] = test['paraphrases'].apply(lambda x: x.split('\',\'') if isinstance(x, str) else x)
# test['paraphrases'] = test['paraphrases'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# ============= CASE : English dataset =============
# import ast  # to safely evaluate string lists

# # Convert string representation of lists to actual lists
# # test['category'] = test['category'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
# test['aspectTerms'] = test['aspectTerms'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# # Display the updated DataFrame to verify the changes
# print(type(test.iloc[1]['aspectTerms']))
# display(test.head())

def clean_list_values(x):
    """
    Clean a cell that may be a list or a single value.
    - If list: remove/skip NaN-like items.
    - If single: return [] if NaN-like, else wrap in a list.
    """
    # Case 1: list
    if isinstance(x, list):
        cleaned = []
        for v in x:
            if v is None or pd.isna(v) or str(v).strip().lower() == "nan":
                continue
            cleaned.append(str(v).strip())
        return cleaned

    # Case 2: scalar value
    if x is None or pd.isna(x) or str(x).strip().lower() == "nan":
        return []  # normalize to empty list
    return [str(x).strip()]  # wrap single value into list
    import ast
def safe_literal_eval(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        return [x] # Return as list containing the string if evaluation fails

# col = ["baseline"	, "explicit_iterative" ,	"implicit_kg"	, "combined_aspects"]

# for c in col:
    # new_col_name= 'cleaned_' + c
    # test[new_col_name] = test[c].apply(safe_literal_eval)


def process_data(df):
    # Convert string representation of lists to actual lists
    df['aspect'] = df['aspect'].apply(safe_literal_eval)
    df['aspect'] = df['aspect'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
    df['paraphrases'] = df['paraphrases'].apply(clean_paraphrases)
    df["aspect"] = df["aspect"].apply(clean_list_values)

    #reset index for test:
    df.reset_index(drop=True, inplace=True)
    return df




time: 3.78 ms (started: 2026-05-03 11:25:24 +00:00)


time: 8.03 ms (started: 2026-05-03 11:25:24 +00:00)


In [ ]:
import ast

def fast_parse_list(x):
    """Parse string/list values, and split by newline if needed."""
    if isinstance(x, str):
        x_strip = x.strip()
        # Try parse if it looks like a Python list
        if x_strip.startswith("[") and x_strip.endswith("]"):
            try:
                parsed = ast.literal_eval(x_strip)
                return fast_parse_list(parsed)  # handle nested
            except Exception:
                return [s.strip() for s in x_strip.split("\n") if s.strip()]
        # Otherwise, split by newline directly
        return [s.strip() for s in x_strip.split("\n") if s.strip()]

    elif isinstance(x, list):
        # flatten, split by newline if needed
        out = []
        for i in x:
            if isinstance(i, str):
                out.extend([s.strip() for s in i.split("\n") if s.strip()])
        return out

    else:
        return [str(x)]

def process_data2(df):
    df = df.copy()
    df["aspect"] = df["aspect"].apply(fast_parse_list)
    df["paraphrases"] = df["paraphrases"].apply(fast_parse_list)
    df.reset_index(drop=True, inplace=True)
    return df


time: 2.06 ms (started: 2026-05-03 11:25:24 +00:00)


In [ ]:
# ─── CAMeL-based normalization (no manual prefix/suffix stripping) ────────────
# Uses camel_tools utilities to safely unify letter forms and strip diacritics.
# We intentionally do NOT strip prefixes/suffixes here: stripping ف corrupts
# فندق→ندق, فريق→ريق, etc.  Morphological matching is done via get_lemmas().
try:
    from camel_tools.utils.normalize import (
        normalize_alef_ar,           # أ إ آ → ا
        normalize_tah_marbuta_ar,    # ة → ه
        normalize_alef_maksura_ar,   # ى → ي
    )
    from camel_tools.utils.dediac import dediac_ar
    def normalize_phrase(phrase: str) -> str:
        """CAMeL-based Arabic normalization — no prefix/suffix stripping."""
        t = str(phrase).strip()
        if not t:
            return t
        t = dediac_ar(t)
        t = normalize_alef_ar(t)
        t = normalize_tah_marbuta_ar(t)
        t = normalize_alef_maksura_ar(t)
        return t.strip()
    def normalize_token(tok: str) -> str:
        return normalize_phrase(tok)
    _NORMALIZE_BACKEND = "camel_tools"
except ImportError:
    import re
    _DIACRITICS = re.compile(r'[\u0617-\u061A\u064B-\u0652]')
    def normalize_phrase(phrase: str) -> str:
        """Fallback: manual letter unification, NO prefix/suffix stripping."""
        t = str(phrase).strip()
        for a, b in [("أ","ا"),("إ","ا"),("آ","ا"),("ى","ي"),("ة","ه"),("ؤ","و"),("ئ","ي")]:
            t = t.replace(a, b)
        t = _DIACRITICS.sub("", t).replace("ـ", "")
        return t.strip()
    def normalize_token(tok: str) -> str:
        return normalize_phrase(tok)
    _NORMALIZE_BACKEND = "fallback"
print(f"  normalize_phrase backend: {_NORMALIZE_BACKEND}")

import re
from typing import List, Optional

# --------- Heuristic adjective filter (fallback) ----------
_ADJ_LEXICON = set("""
جيد جيدة ممتاز ممتازة سيئ سيئة رائع رائعة جميل جميلة كبير كبيرة صغير صغيرة طويل طويلة قصير قصيرة
سريع سريعة بطيء بطيئة مجاني مجانية جديد جديدة قديم قديمة طبيعي طبيعية واضح واضحة قوي قوية ضعيف ضعيفة
""".split())

def _looks_like_adj(word: str) -> bool:
    """Very light guess: nisba endings + small lexicon."""
    w = word.strip()
    if not w:
        return False
    # common nisba endings and feminine endings (rough heuristic)
    if w.endswith(("ي", "ية", "يات", "ان", "ين")):
        return True
    if w in _ADJ_LEXICON:
        return True
    return False

# --------- CAMeL-based adjective remover (accurate) ----------
class CamelAdjRemover:
    """Remove adjectives from a phrase using CAMeL Tools POS tags."""
    def __init__(self):
        from camel_tools.tokenizers.word import simple_word_tokenize
        from camel_tools.disambig.mle import MLEDisambiguator
        self.tok = simple_word_tokenize
        self.mle = MLEDisambiguator.pretrained()
    def strip_adjectives(self, phrase: str) -> str:
        tokens = self.tok(phrase)
        disamb = self.mle.disambiguate(tokens)
        kept = []
        for t, d in zip(tokens, disamb):
            pos = None
            if d.analyses:
                # POS can appear as 'pos' or 'pos_tag' depending on version
                a = d.analyses[0]
                pos = (getattr(a, "pos", None) or getattr(a, "pos_tag", "") or "").lower()
            # keep if not adjective
            if not (pos.startswith("adj") or pos == "a"):
                kept.append(t)
        return " ".join(kept).strip()

# --------- Heuristic phrase cleaner (no CAMeL) ----------
def strip_adjectives_heuristic(phrase: str) -> str:
    toks = [t for t in re.split(r"\s+", phrase.strip()) if t]
    kept = [t for t in toks if not _looks_like_adj(t)]
    return " ".join(kept).strip()

# --------- Normalization (reuse from earlier) ----------



# --------- Parser (reuse, with dedup) ----------
_SEP_PAT = re.compile(r"\s*(?:\[sep\]|\[SEP\]|،|,|;|؛|\||\n|\t|\/|\\|:|\s{2,})\s*", re.IGNORECASE)
def parse_aspect_string(s) -> List[str]:
    if s is None:
        return []
    if isinstance(s, list):
        items = s
    else:
        s = str(s).strip().strip("[](){}").replace("'", "").replace('"', "")
        items = _SEP_PAT.split(s)
    cleaned = []
    for t in items:
        t = re.sub(r"^[\s\-\–\—\·\•\*\.]+|[\s\-\–\—\·\•\*\.]+$", "", t.strip())
        if t:
            cleaned.append(t)
    # dedup by normalized form, but keep first surface
    seen_norm = set()
    out = []
    for w in cleaned:
        key = normalize_phrase(w)
        if key and key not in seen_norm:
            seen_norm.add(key)
            out.append(w)
    return out

# --------- Main cleaner for predictions ----------
def clean_predicted_aspects(raw_pred: str,
                            use_camel: bool = True,
                            camel_obj: Optional[CamelAdjRemover] = None) -> List[str]:
    """
    1) parse string to list
    2) remove adjectives (CAMeL if available, else heuristic)
    3) normalize for dedup
    4) drop empties
    """
    aspects = parse_aspect_string(raw_pred)

    if use_camel:
        try:
            camel = camel_obj or CamelAdjRemover()
            stripped = [camel.strip_adjectives(a) for a in aspects]
        except Exception:
            # fallback if camel-tools missing/unavailable
            stripped = [strip_adjectives_heuristic(a) for a in aspects]
    else:
        stripped = [strip_adjectives_heuristic(a) for a in aspects]

    # drop empties and dedup (by normalized)
    seen = set()
    final = []
    for a in stripped:
        a = a.strip()
        if not a:
            continue
        key = normalize_phrase(a)
        if key and key not in seen:
            seen.add(key)
            final.append(a)
    return final

# ─── Guard A: Morphological containment via CAMeL lemmas ─────────────────────
_MLE_SINGLETON  = None
_TOK_SINGLETON  = None

def _get_mle():
    global _MLE_SINGLETON, _TOK_SINGLETON
    if globals().get('_MLE_SINGLETON') is None:
        try:
            from camel_tools.disambig.mle import MLEDisambiguator
            from camel_tools.tokenizers.word import simple_word_tokenize
            _MLE_SINGLETON = MLEDisambiguator.pretrained()
            _TOK_SINGLETON = simple_word_tokenize
        except Exception:
            _MLE_SINGLETON = False
    return (None, None) if _MLE_SINGLETON is False else (_MLE_SINGLETON, _TOK_SINGLETON)

def get_lemmas(text: str) -> set:
    """Return the set of normalised lemmas for all tokens in text."""
    mle, tok = _get_mle()
    if mle is not None:
        try:
            tokens = tok(str(text))
            disamb = mle.disambiguate(tokens)
            lemmas = set()
            for d in disamb:
                if d.analyses:
                    lem = getattr(d.analyses[0], "lemma", None)
                    if lem:
                        lemmas.add(normalize_phrase(str(lem)))
            if lemmas:
                return lemmas
        except Exception:
            pass
    # Fallback: normalised surface tokens
    return set(normalize_phrase(w) for w in str(text).split() if w)

# Generic nouns that appear in almost every hotel/restaurant sentence and are
# almost always added spuriously by IG rounds 2+. Only kept if round 1
# already predicted them.
# Both ال-prefixed and bare forms so Guard A blocks them regardless of
# which surface form the model produces (الفندق vs فندق, etc.)
_GENERIC_DOMAIN_NOUNS = {
    normalize_phrase(w) for w in [
        "فندق", "الفندق", "فنادق", "الفنادق",
        "مطعم", "المطعم", "مطاعم", "المطاعم",
        "موقع", "الموقع", "موقعه", "موقعها",
        "المكان", "مكان",
        "الخدمة", "خدمة", "الخدمه", "خدمه",
        "الغرفة", "غرفة", "الغرفه", "غرفه",
        "السعر", "سعر",
        "المنطقة", "المنطقه",
    ]
}

def aspect_grounded_in_sentence(
    aspect: str,
    original_sentence: str,
    round1_aspects: list = None,
) -> bool:
    """
    Guard A – True if any lemma of 'aspect' appears in the lemmatized sentence.

    Extra rules applied before the lemma check:
    1. Fragments ≤2 normalised chars are always rejected.
    2. Generic domain nouns (فندق, مطعم …) are rejected in round 2+
       UNLESS round 1 already predicted them — this stops the model from
       filling in obvious domain keywords after partial masking.
    """
    norm_asp = normalize_phrase(aspect)

    # Rule 1: too short to be a real aspect
    if len(norm_asp) <= 2:
        return False

    # Rule 2: generic domain noun not seen in round 1
    if round1_aspects is not None:
        round1_norms = {normalize_phrase(a) for a in round1_aspects}
        if norm_asp in _GENERIC_DOMAIN_NOUNS and norm_asp not in round1_norms:
            return False

    asp_lemmas  = get_lemmas(aspect)
    sent_lemmas = get_lemmas(original_sentence)
    return bool(asp_lemmas & sent_lemmas)

# ─── Guard B: Semantic novelty via multilingual SBERT ────────────────────────
_SBERT_GUARD = None

def _get_sbert_guard():
    global _SBERT_GUARD
    if globals().get('_SBERT_GUARD') is None:
        try:
            from sentence_transformers import SentenceTransformer
            _SBERT_GUARD = SentenceTransformer(
                "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
            )
        except Exception:
            _SBERT_GUARD = False
    return None if _SBERT_GUARD is False else _SBERT_GUARD

def is_novel_aspect(candidate: str, reference_aspects: list,
                    novelty_threshold: float = 0.85) -> bool:
    """
    Guard B – True if 'candidate' is semantically distinct from all
    reference_aspects (i.e. it was not already captured in round 1).
    Threshold 0.85: aspects above this are considered paraphrases/duplicates.
    """
    if not reference_aspects:
        return True
    sbert = _get_sbert_guard()
    if sbert is None:
        # Fallback: normalised-surface overlap
        norm_c   = normalize_phrase(candidate)
        norm_refs = [normalize_phrase(r) for r in reference_aspects]
        return norm_c not in norm_refs
    try:
        from sentence_transformers import util
        cand_emb = sbert.encode(candidate,         convert_to_tensor=True)
        ref_embs = sbert.encode(reference_aspects, convert_to_tensor=True)
        sims     = util.cos_sim(cand_emb, ref_embs).cpu().numpy().flatten()
        return float(sims.max()) < novelty_threshold
    except Exception:
        return True   # fail-open: let it through


  normalize_phrase backend: fallback
time: 8.57 ms (started: 2026-05-03 11:25:24 +00:00)


In [ ]:
# ─── CAMeL-based normalization (no manual prefix/suffix stripping) ────────────
# Uses camel_tools utilities to safely unify letter forms and strip diacritics.
# We intentionally do NOT strip prefixes/suffixes here: stripping ف corrupts
# فندق→ندق, فريق→ريق, etc.  Morphological matching is done via get_lemmas().
try:
    from camel_tools.utils.normalize import (
        normalize_alef_ar,           # أ إ آ → ا
        normalize_tah_marbuta_ar,    # ة → ه
        normalize_alef_maksura_ar,   # ى → ي
    )
    from camel_tools.utils.dediac import dediac_ar
    def normalize_phrase(phrase: str) -> str:
        """CAMeL-based Arabic normalization — no prefix/suffix stripping."""
        t = str(phrase).strip()
        if not t:
            return t
        t = dediac_ar(t)
        t = normalize_alef_ar(t)
        t = normalize_tah_marbuta_ar(t)
        t = normalize_alef_maksura_ar(t)
        return t.strip()
    def normalize_token(tok: str) -> str:
        return normalize_phrase(tok)
    _NORMALIZE_BACKEND = "camel_tools"
except ImportError:
    import re
    _DIACRITICS = re.compile(r'[\u0617-\u061A\u064B-\u0652]')
    def normalize_phrase(phrase: str) -> str:
        """Fallback: manual letter unification, NO prefix/suffix stripping."""
        t = str(phrase).strip()
        for a, b in [("أ","ا"),("إ","ا"),("آ","ا"),("ى","ي"),("ة","ه"),("ؤ","و"),("ئ","ي")]:
            t = t.replace(a, b)
        t = _DIACRITICS.sub("", t).replace("ـ", "")
        return t.strip()
    def normalize_token(tok: str) -> str:
        return normalize_phrase(tok)
    _NORMALIZE_BACKEND = "fallback"
print(f"  normalize_phrase backend: {_NORMALIZE_BACKEND}")

import re
from typing import List, Tuple, Dict, Iterable

# -----------------------------
# 1) Parse model string -> list
# -----------------------------
_SEP_PAT = re.compile(
    r"\s*(?:\[sep\]|\[SEP\]|،|,|;|؛|\||\n|\t|\/|\\|:|\s{2,})\s*", re.IGNORECASE
)

def parse_aspect_string(s) -> List[str]:
    """
    Robustly parse a model's string into a list of aspect terms.
    Handles Arabic/English commas, ; ؛ | [sep], brackets, quotes, whitespace.
    If input is already a list, returns a cleaned version.
    """
    if s is None:
        return []
    if isinstance(s, list):
        items = s
    else:
        # strip surrounding brackets/quotes if present
        s = str(s).strip()
        s = s.strip("[](){}")
        s = s.replace("'", "").replace('"', "")
        # split on many possible separators
        items = _SEP_PAT.split(s)

    # clean tokens: remove empties/stray punctuation
    cleaned = []
    for t in items:
        t = t.strip()
        if not t:
            continue
        # remove leading/trailing punctuation
        t = re.sub(r"^[\s\-\–\—\·\•\*\.]+|[\s\-\–\—\·\•\*\.]+$", "", t)
        if t:
            cleaned.append(t)
    # de-duplicate while preserving order
    seen = set()
    dedup = []
    for w in cleaned:
        if w not in seen:
            seen.add(w)
            dedup.append(w)
    return dedup

# ----------------------------------------
# 2) Light Arabic normalization for eval
# ----------------------------------------
# common clitic prefixes and pronominal suffixes



# ----------------------------------------------------
# 3) Flexible matching: exact, substring, Jaccard(IoU)
# ----------------------------------------------------
def jaccard(a: str, b: str) -> float:
    A = set(a.split())
    B = set(b.split())
    if not A and not B:
        return 1.0
    if not A or not B:
        return 0.0
    return len(A & B) / len(A | B)

def similarity(gold: str, pred: str) -> float:
    """Return similarity score in [0,1] after normalization.
    1.0 if exact or substring; else token Jaccard."""
    g = normalize_phrase(gold)
    p = normalize_phrase(pred)
    if not g or not p:
        return 0.0
    if g == p:
        return 1.0
    if g in p or p in g:
        return 1.0
    return jaccard(g, p)

# ---------------------------------------------------
# 4) Greedy bipartite matching above a threshold
# ---------------------------------------------------
def best_alignment(gold: List[str], pred: List[str], thresh: float = 0.6) -> Tuple[int, List[Tuple[int,int,float]]]:
    """
    Greedy maximum-weight matching:
    - Build all pairs with sim >= thresh
    - Sort by similarity desc
    - Pick non-conflicting pairs
    Returns TP count and list of matches (gi, pj, sim)
    """
    pairs = []
    for i, g in enumerate(gold):
        for j, p in enumerate(pred):
            s = similarity(g, p)
            if s >= thresh:
                pairs.append((i, j, s))
    pairs.sort(key=lambda x: x[2], reverse=True)

    matched_g = set()
    matched_p = set()
    matches = []
    for gi, pj, s in pairs:
        if gi not in matched_g and pj not in matched_p:
            matched_g.add(gi)
            matched_p.add(pj)
            matches.append((gi, pj, s))
    return len(matches), matches

# ----------------------------------------------------------------
# 5) Precision/Recall/F1 for a single example and aggregated stats
# ----------------------------------------------------------------
# def prf1_for_example(gold_terms: Iterable[str], pred_terms: Iterable[str], thresh: float = 0.6) -> Dict[str, float]:
#     gold = [t for t in map(str, gold_terms) if t and t.strip()]
#     pred = [t for t in map(str, pred_terms) if t and t.strip()]

#     tp, _ = best_alignment(gold, pred, thresh=thresh)
#     fp = max(0, len(pred) - tp)
#     fn = max(0, len(gold) - tp)

#     prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
#     rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
#     f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0

#     return {"tp": tp, "fp": fp, "fn": fn, "precision": prec, "recall": rec, "f1": f1}

def prf1_for_example(gold_terms: Iterable[str], pred_terms: Iterable[str], thresh: float = 0.6) -> Dict[str, float]:
    gold = [t for t in map(str, gold_terms) if t and t.strip()]
    pred = [t for t in map(str, pred_terms) if t and t.strip()]

    # Special case: both empty → perfect prediction
    if not gold and not pred:
        return {"tp": 0, "fp": 0, "fn": 0,
                "precision": 1.0, "recall": 1.0, "f1": 1.0}

    tp, _ = best_alignment(gold, pred, thresh=thresh)
    fp = max(0, len(pred) - tp)
    fn = max(0, len(gold) - tp)

    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0

    return {"tp": tp, "fp": fp, "fn": fn,
            "precision": prec, "recall": rec, "f1": f1}


def evaluate_aspects(
    gold_lists: List[List[str]],
    pred_strings: List[str],
    parse_threshold: float = 0.6
) -> Dict[str, object]:
    """
    Evaluate lists of gold aspects vs model string predictions.
    - gold_lists: list of gold lists, e.g., [['الفندق','موقف'], ...]
    - pred_strings: list of raw model strings for each example
    - parse_threshold: similarity threshold for a match (default 0.6)
    Returns dict with micro/macro P/R/F1 and per-example scores.
    """
    assert len(gold_lists) == len(pred_strings), "gold and pred sizes must match"

    per_example = []
    tot_tp = tot_fp = tot_fn = 0

    for gold, pred_str in zip(gold_lists, pred_strings):
        pred_list = parse_aspect_string(pred_str)
        metrics = prf1_for_example(gold, pred_list, thresh=parse_threshold)
        per_example.append(metrics)
        tot_tp += metrics["tp"]
        tot_fp += metrics["fp"]
        tot_fn += metrics["fn"]

    # micro
    micro_p = tot_tp / (tot_tp + tot_fp) if (tot_tp + tot_fp) > 0 else 0.0
    micro_r = tot_tp / (tot_tp + tot_fn) if (tot_tp + tot_fn) > 0 else 0.0
    micro_f = 2 * micro_p * micro_r / (micro_p + micro_r) if (micro_p + micro_r) > 0 else 0.0

    # macro
    if per_example:
        macro_p = sum(x["precision"] for x in per_example) / len(per_example)
        macro_r = sum(x["recall"] for x in per_example) / len(per_example)
        macro_f = sum(x["f1"] for x in per_example) / len(per_example)
    else:
        macro_p = macro_r = macro_f = 0.0

    return {
        "micro": {"precision": micro_p, "recall": micro_r, "f1": micro_f, "tp": tot_tp, "fp": tot_fp, "fn": tot_fn},
        "macro": {"precision": macro_p, "recall": macro_r, "f1": macro_f},
        "per_example": per_example,
    }


  normalize_phrase backend: fallback
time: 9.58 ms (started: 2026-05-03 11:25:24 +00:00)


In [ ]:
import json
from pathlib import Path
import networkx as nx
from typing import Dict, Any, List

class KnowledgeGraph:
    """
    Reusable Knowledge Graph wrapper (NetworkX).
    - Handles both structured aspects (dicts) and string lists.
    - Normalizes IDs but preserves original labels.
    - IMPORTANT: Ignores any JSON edges of the form aspect -> is_relevant_to_domain -> domain.
                 Only uses:
                   adjective -> describes_aspect -> aspect
                   aspect    -> belongs_to_entity -> entity
                   entity    -> is_relevant_to_domain -> domain
    """
    def __init__(self):
        self.G = nx.DiGraph()
        self.meta: Dict[str, Any] = {}
        # Quick indexes for convenience
        self._id2label: Dict[str, str] = {}
        self._label2id: Dict[str, str] = {}

    @staticmethod
    def _norm(s):
        return s.strip().lower() if isinstance(s, str) else s

    @staticmethod
    def _pred_str(d) -> str:
        return (d.get("predicate") or "").lower().replace(" ", "_")

    @classmethod
    def from_json(cls, path: str | Path) -> "KnowledgeGraph":
        kg = cls()
        kg._load(Path(path))
        return kg

    def _add_node(self, nid: str, label: str, ntype: str, **attrs):
        if nid not in self.G:
            self.G.add_node(nid, type=ntype, label=label, **attrs)
            if label and label not in self._label2id:
                self._label2id[label] = nid
            self._id2label[nid] = label
        else:
            # upgrade type if unknown
            if ntype and self.G.nodes[nid].get("type") == "unknown":
                self.G.nodes[nid]["type"] = ntype
            if "label" not in self.G.nodes[nid]:
                self.G.nodes[nid]["label"] = label

    def _load(self, path: Path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        self.meta["source_path"] = str(path)

        # 1) Domains
        for d in data.get("domains", []):
            if isinstance(d, dict) and "name" in d:
                did = self._norm(d["name"])
                self._add_node(did, d["name"], "domain", **{k: v for k, v in d.items() if k != "name"})

        # 2) Entities (+ entity -> domain)
        for e in data.get("entities", []):
            if isinstance(e, dict) and "name" in e:
                eid = self._norm(e["name"])
                self._add_node(eid, e["name"], "entity", **{k: v for k, v in e.items() if k != "name"})
                dom = e.get("domain")
                if isinstance(dom, str) and dom.strip():
                    did = self._norm(dom)
                    if did not in self.G:
                        self._add_node(did, dom, "domain")
                    self.G.add_edge(
                        eid, did,
                        predicate="is_relevant_to_domain",
                        subject=e["name"], object=dom,
                        sub_lang=e.get("language"), obj_lang=e.get("language")
                    )

        # 3) Aspects (+ aspect -> entity)
        aspects = data.get("aspects", [])
        if aspects and isinstance(aspects[0], dict):
            for a in aspects:
                if not isinstance(a, dict) or "name" not in a:
                    continue
                aid = self._norm(a["name"])
                self._add_node(aid, a["name"], "aspect", **{k: v for k, v in a.items() if k != "name"})
                ents = a.get("entity", []) or []
                for ent in ents:
                    eid = self._norm(ent)
                    if eid not in self.G:
                        self._add_node(eid, ent, "entity")
                    self.G.add_edge(
                        aid, eid,
                        predicate="belongs_to_entity",
                        subject=a["name"], object=ent,
                        sub_lang=a.get("language"), obj_lang=a.get("language")
                    )
        else:
            # Aspects as strings
            for a in aspects:
                if isinstance(a, str):
                    aid = self._norm(a)
                    self._add_node(aid, a, "aspect")

        # 4) Adjectives
        for adj in data.get("adjectives", []):
            if isinstance(adj, str):
                aid = self._norm(adj)
                self._add_node(aid, adj, "adjective")

        # 5) Relationships (ignore aspect -> domain)
        for rel in data.get("relationships", []):
            subj = self._norm(rel.get("subject"))
            obj  = self._norm(rel.get("object"))
            pred = self._pred_str(rel)
            if not subj or not obj or not pred:
                continue

            # ensure nodes exist
            if subj not in self.G:
                self._add_node(subj, rel.get("subject", subj), "unknown")
            if obj not in self.G:
                self._add_node(obj, rel.get("object", obj), "unknown")

            # HARD RULE: JSON must NOT define aspect -> is_relevant_to_domain -> domain
            if pred == "is_relevant_to_domain" and self.G.nodes[subj].get("type") == "aspect":
                continue

            # add allowed edge as-is
            self.G.add_edge(subj, obj, **rel)

        # Summary
        self.meta["counts"] = self.counts()

    # ------------------- Queries -------------------
    def counts(self) -> dict:
        types = {}
        for _, d in self.G.nodes(data=True):
            t = d.get("type", "unknown")
            types[t] = types.get(t, 0) + 1
        return {
            "nodes": self.G.number_of_nodes(),
            "edges": self.G.number_of_edges(),
            **{f"n_{k}": v for k, v in types.items()},
        }

    def get_traces_for_adjective(self, adjective: str) -> List[dict]:
        """Return all adjective -> aspect -> entity -> domain traces."""
        adj_id = self._norm(adjective)
        if adj_id not in self.G:
            return []
        traces = []
        for _, aspect_id, d1 in self.G.out_edges(adj_id, data=True):
            if self._pred_str(d1) != "describes_aspect":
                continue
            for _, ent_id, d2 in self.G.out_edges(aspect_id, data=True):
                if self._pred_str(d2) != "belongs_to_entity":
                    continue
                for _, dom_id, d3 in self.G.out_edges(ent_id, data=True):
                    if self._pred_str(d3) != "is_relevant_to_domain":
                        continue
                    traces.append({
                        "adjective": self.G.nodes[adj_id].get("label", adj_id),
                        "aspect":    self.G.nodes[aspect_id].get("label", aspect_id),
                        "entity":    self.G.nodes[ent_id].get("label", ent_id),
                        "domain":    self.G.nodes[dom_id].get("label", dom_id),
                    })
        return traces

    def get_aspects_by_entity(self, entity: str) -> List[str]:
        """Aspects that belong to a given entity."""
        eid = self._norm(entity)
        if eid not in self.G:
            return []
        aspects = [
            u for u, v, d in self.G.in_edges(eid, data=True)
            if self._pred_str(d) == "belongs_to_entity"
        ]
        return sorted({self.G.nodes[a].get("label", a) for a in aspects})

    def get_entities_by_domain(self, domain: str) -> List[str]:
        """Entities that are relevant to a given domain."""
        did = self._norm(domain)
        if did not in self.G:
            return []
        ents = [
            u for u, v, d in self.G.in_edges(did, data=True)
            if self._pred_str(d) == "is_relevant_to_domain"
        ]
        return sorted({self.G.nodes[e].get("label", e) for e in ents})

    def get_adjectives_for_aspect(self, aspect: str) -> List[str]:
        """Adjectives that describe a given aspect."""
        aid = self._norm(aspect)
        if aid not in self.G:
            return []
        adjs = [
            u for u, v, d in self.G.in_edges(aid, data=True)
            if self._pred_str(d) == "describes_aspect"
        ]
        return sorted({self.G.nodes[x].get("label", x) for x in adjs})

    # ---------- NEW convenience helpers (names as requested) ----------
    def get_aspects(self, adjective: str) -> List[str]:
        """Aspects linked from an adjective via 'describes_aspect'."""
        adj_id = self._norm(adjective)
        if adj_id not in self.G:
            return []
        aspects = [
            v for _, v, d in self.G.out_edges(adj_id, data=True)
            if self._pred_str(d) == "describes_aspect"
        ]
        return sorted({self.G.nodes[a].get("label", a) for a in aspects})

    def get_entities(self, aspect: str) -> List[str]:
        """Entities linked from an aspect via 'belongs_to_entity'."""
        aid = self._norm(aspect)
        if aid not in self.G:
            return []
        ents = [
            v for _, v, d in self.G.out_edges(aid, data=True)
            if self._pred_str(d) == "belongs_to_entity"
        ]
        return sorted({self.G.nodes[e].get("label", e) for e in ents})

    def get_domains(self, entity: str) -> List[str]:
        """Domains linked from an entity via 'is_relevant_to_domain'."""
        eid = self._norm(entity)
        if eid not in self.G:
            return []
        doms = [
            v for _, v, d in self.G.out_edges(eid, data=True)
            if self._pred_str(d) == "is_relevant_to_domain"
        ]
        return sorted({self.G.nodes[d].get("label", d) for d in doms})

    # ------------------- Persistence -------------------
    def save_pickle(self, path: str | Path):
        import pickle
        with open(path, "wb") as f:
            pickle.dump({"graph": self.G, "meta": self.meta}, f)

    @classmethod
    def load_pickle(cls, path: str | Path) -> "KnowledgeGraph":
        import pickle
        with open(path, "rb") as f:
            payload = pickle.load(f)
        kg = cls()
        kg.G = payload["graph"]
        kg.meta = payload.get("meta", {})
        # rebuild label caches
        for n, d in kg.G.nodes(data=True):
            label = d.get("label", n)
            kg._id2label[n] = label
            if label not in kg._label2id:
                kg._label2id[label] = n
        return kg


time: 7.76 ms (started: 2026-05-03 11:25:24 +00:00)


In [ ]:
# def get_camel() -> CamelAdjRemover:
#     global _CAMEL_SINGLETON
#     if _CAMEL_SINGLETON is None:
#         _CAMEL_SINGLETON = CamelAdjRemover()
#     return _CAMEL_SINGLETON
def get_camel() -> CamelAdjRemover:
    global _CAMEL_SINGLETON
    if globals().get('_CAMEL_SINGLETON') is None:
        _CAMEL_SINGLETON = CamelAdjRemover()
    return _CAMEL_SINGLETON

time: 521 µs (started: 2026-05-03 11:25:24 +00:00)


### Inference:

#### Inference function:

In [ ]:

import re


FastLanguageModel.for_inference(model) # Enable native 2x faster inference


prompt = """Below is an instruction that describes a task, paired with an input that provides a text. Write a response that appropriately completes the request.
###Instruction:
{}
###Text:
{}
###Aspect:
{}"""

# instruction = "Act as a Natural Language processing expert in Arabic language, your task is to to extract the aspect term/terms that are being described in the following sentence. Mostly, the identified aspect can be a noun or a noun phrase, your answer should contain the extracted aspects in Arabic."
instruction = """Act as a Natural Language Processing expert in Arabic language.
Your task is to extract aspect term(s) that are explicitly mentioned in the given sentence.
- Only return aspect terms that appear verbatim in the sentence.
- Do not infer or invent related words.
- If no aspect is found, return "none"."""

def extract_aspect_from_response(text_list):
    if not text_list or not isinstance(text_list, list):
        return []
    model_output = text_list[0]  # get the string inside the list
    try:
        aspect_part = model_output.split("###Aspect:\n", 1)[1]
    except IndexError:
        print('index error')
        return []
    aspect = aspect_part.split("<|end_of_text|>", 1)[0].strip()

    # Try to interpret model output as Python list, else fallback to single term
    try:
        parsed = ast.literal_eval(aspect)
        if isinstance(parsed, list):
            return [a.strip() for a in parsed if a.strip()]
    except Exception:
        pass

    return [aspect] if aspect else []

def extract_aspect_from_response2(text_list):
    # text_list is a list, expected to have one string element
    if not text_list or not isinstance(text_list, list):
        return None
    model_output = text_list[0]  # get the string inside the list
    try:
       aspect_part = model_output.split("###Aspect:\n", 1)[1]
    except IndexError:
        print('index error')
        return None
    aspect = aspect_part.split("<|end_of_text|>", 1)[0].strip()
    return aspect

EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN


def extract_aspect(sentence):

    inputs = tokenizer(
    [
        prompt.format(
            instruction, sentence, # input
            "", # output - leave this blank for generation!
        )#+ EOS_TOKEN
    ], return_tensors = "pt").to("cuda")

    outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)
    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=False)
    aspect_terms = extract_aspect_from_response(decoded)
    #print(f'Sentence: {sentence}')
    #print(f'Model prediction: {aspect_terms}')
    return(aspect_terms)



time: 16.9 ms (started: 2026-05-03 11:25:24 +00:00)


#### 1. Iterative Residual Decoding:

In [ ]:
def iterative_residual_decoding(sentence: str, max_rounds: int = 3,
                                novelty_threshold: float = 0.85) -> list:
    """
    Iterative Generation (IG) — used for both English and Arabic.

    Fixes:
    1. \b regex replaced with str.replace() (Arabic-safe).
    2. Guard A: round-2+ aspects must be lemma-grounded in original sentence.
    3. Guard B: round-2+ aspects must be semantically novel vs round-1 aspects.
    """
    original_sentence  = sentence
    remaining_sentence = sentence
    all_aspects        = []
    seen_norm          = set()
    round1_aspects     = []

    for round_id in range(max_rounds):
        aspects = extract_aspect(remaining_sentence)
        if not aspects:
            break

        new_aspects = [a.strip() for a in aspects
                       if a and a.strip() and a.strip().lower() != "none"]
        if not new_aspects:
            break

        for asp in new_aspects:
            norm = normalize_phrase(asp)
            if not norm or norm in seen_norm:
                continue

            # ── Round 2+ guards ────────────────────────────────────────────
            if round_id > 0:
                if not aspect_grounded_in_sentence(asp, original_sentence, round1_aspects):
                    continue
                if not is_novel_aspect(asp, round1_aspects, novelty_threshold):
                    continue

            seen_norm.add(norm)
            all_aspects.append(asp)
            remaining_sentence = remaining_sentence.replace(asp, "").strip()

        if round_id == 0:
            round1_aspects = [a for a in all_aspects if a.lower() != "none"]

        if not remaining_sentence:
            break

    return [a for a in all_aspects if a.lower() != "none"]


time: 1.43 ms (started: 2026-05-03 11:25:24 +00:00)


In [ ]:

# example = test.iloc[104]
# prediction= iterative_residual_decoding(example['sentence'])
# print('true aspect:', example['aspect'], '            Model Prediction:', prediction)

time: 220 µs (started: 2026-05-03 11:25:24 +00:00)


#### 2. Input transformation through paraphrasing:

In [ ]:
import pandas as pd
import math
import re
from collections import Counter
from sentence_transformers import SentenceTransformer, util

# ─────────────────────────────────────────────────────────────────────────────
# BUG FIX 1: use multilingual SBERT – all-MiniLM-L6-v2 is English-only
#            and produces garbage embeddings for Arabic.
# ─────────────────────────────────────────────────────────────────────────────
_sbert = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# ─────────────────────────────────────────────────────────────────────────────
# BUG FIX 2: Arabic noun-head extraction with CAMeL Tools
#            Fallback to last-token of normalized phrase when CAMeL unavailable.
# ─────────────────────────────────────────────────────────────────────────────
_CAMEL_MLE_AR  = None   # lazy singleton
_CAMEL_TOK_AR  = None

def _get_camel_for_noun_head():
    """Lazy load CAMeL MLE disambiguator (returns (mle, tok) or (None, None))."""
    global _CAMEL_MLE_AR, _CAMEL_TOK_AR
    if _CAMEL_MLE_AR is None:
        try:
            from camel_tools.disambig.mle import MLEDisambiguator
            from camel_tools.tokenizers.word import simple_word_tokenize
            _CAMEL_MLE_AR  = MLEDisambiguator.pretrained()
            _CAMEL_TOK_AR  = simple_word_tokenize
        except Exception:
            _CAMEL_MLE_AR  = False   # sentinel: unavailable
            _CAMEL_TOK_AR  = False
    if _CAMEL_MLE_AR is False:
        return None, None
    return _CAMEL_MLE_AR, _CAMEL_TOK_AR

def _noun_head_lemma_ar(phrase: str) -> str:
    """
    Arabic noun-head using CAMeL MLE POS tags.
    Fallback: last whitespace token of the normalized phrase.
    This replaces the English spaCy _noun_head_lemma() which produces
    nonsense on Arabic text.
    """
    phrase = str(phrase).strip()
    if not phrase:
        return phrase

    mle, tok = _get_camel_for_noun_head()
    if mle is not None:
        try:
            tokens = tok(phrase)
            disamb  = mle.disambiguate(tokens)
            nouns   = []
            for t, d in zip(tokens, disamb):
                if d.analyses:
                    pos = (getattr(d.analyses[0], "pos", None)
                           or getattr(d.analyses[0], "pos_tag", "") or "").lower()
                    if pos.startswith("n"):   # noun / proper noun
                        nouns.append(t)
            if nouns:
                return nouns[-1]   # head = last noun token
        except Exception:
            pass  # fall through to verbatim fallback

    # Verbatim / normalized fallback: last token after light normalization
    n = normalize_phrase(phrase)
    parts = [w for w in n.split() if w]
    return parts[-1] if parts else n

# Alias used throughout this notebook (replaces old English version)
def _noun_head_lemma(text: str) -> str:
    return _noun_head_lemma_ar(text)


def min_consensus_for_k(k: int) -> int:
    return max(math.floor(k / 3), 1)

def _canonicalize_list(lst):
    return [(_noun_head_lemma(a), a) for a in lst]   # (canon, raw)

def _consensus_on_canonical(aspects_by_input, min_votes=2):
    """Vote using Arabic canonical (noun-head) forms across aspect lists."""
    cnt = Counter()
    for lst in aspects_by_input:
        canon_set = set(_noun_head_lemma(a) for a in lst)
        for c in canon_set:
            cnt[c] += 1
    return {c for c, v in cnt.items() if v >= min_votes}

def _normalize_for_verbatim(s: str) -> str:
    """Light normalization for verbatim containment checks (language-neutral)."""
    s = str(s)
    s = re.sub(r"[\"'`]+", "", s)
    s = re.sub(r"[()\[\]{},;:]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def _explicitness_filter(aspects, original_sentence, jaccard_threshold=0.5):
    """
    Keep aspect if:
      - verbatim substring in original (after light normalization), OR
      - head noun (Arabic noun-head via CAMeL / fallback) appears verbatim, OR
      - token-level Jaccard with original >= threshold.
    """
    sent_norm   = _normalize_for_verbatim(original_sentence)
    sent_tokens = set(sent_norm.split())

    kept = []
    for a in aspects:
        low = _normalize_for_verbatim(a)
        if not low:
            continue
        if low in sent_norm:
            kept.append(a); continue
        head = _noun_head_lemma(a)
        if head and _normalize_for_verbatim(head) in sent_norm:
            kept.append(a); continue
        a_tokens = set(low.split())
        inter = len(a_tokens & sent_tokens)
        union = len(a_tokens | sent_tokens) or 1
        if inter / union >= jaccard_threshold:
            kept.append(a)
    return kept

def _dedup_preserve(seq):
    seen, out = set(), []
    for x in seq:
        k = _normalize_for_verbatim(x)
        if k not in seen and k:
            seen.add(k)
            out.append(x)
    return out


# ─────────────────────────────────────────────────────────────────────────────
# Main paraphrase extraction function
# (uses multilingual SBERT + Arabic noun-head throughout)
# ─────────────────────────────────────────────────────────────────────────────
def extract_aspects_with_paraphrases(
    sentences,
    top_k=2,
    similarity_threshold=0.58,
    include_original_always=True,
    jaccard_explicitness=0.50,
    verbose=False,
):
    """
    sentences : [original, para1, ..., para10]
    Runs iterative_residual_decoding ONCE per input (original + paraphrases),
    caches predictions, then materialises for k in {0,2,4,6,8,10}.
    Returns 1-row DataFrame.
    """
    empty = {
        "pred_orig": [], "pred_para_list": [],
        "k_0Preds": [], "k_2Preds": [], "k_4Preds": [],
        "k_6Preds": [], "k_8Preds": [], "k_10Preds": [],
    }
    if not sentences or not isinstance(sentences, list):
        return pd.DataFrame([empty])

    original_sentence = sentences[0]
    # paraphrases       = sentences[1:][:10]#for 10 paraphrases
    paraphrases       = sentences[1:3] #for two paraphrases.


    # 1) Expensive once: iterative decoding
    per_input = [iterative_residual_decoding(s, max_rounds=2)
                 for s in [original_sentence] + paraphrases]

    pred_orig      = per_input[0]
    pred_para_list = per_input[1:]

    pinned_originals = _dedup_preserve(pred_orig) if include_original_always else []

    def _aggregate_for_k(k: int):
        if k == 0 or not pred_para_list:
            return _dedup_preserve(pinned_originals)

        para_lists_k = pred_para_list[:k]
        mc           = min_consensus_for_k(k)
        voted_canon  = _consensus_on_canonical(para_lists_k, min_votes=mc)

        canon_to_raw = {}
        for lst in para_lists_k:
            for c, raw in _canonicalize_list(lst):
                if c in voted_canon:
                    if c not in canon_to_raw or len(raw) > len(canon_to_raw[c]):
                        canon_to_raw[c] = raw
        candidates = list(canon_to_raw.values())

        if not candidates:
            candidates = list({a for lst in para_lists_k for a in lst})

        kept_from_para = []
        if candidates:
            orig_emb = _sbert.encode(original_sentence, convert_to_tensor=True)
            asp_embs = _sbert.encode(candidates,        convert_to_tensor=True)
            sims     = util.cos_sim(asp_embs, orig_emb).cpu().numpy().flatten().tolist()
            ranked   = sorted(zip(candidates, sims), key=lambda x: x[1], reverse=True)
            ranked   = [(a, s) for a, s in ranked if s >= similarity_threshold]
            if top_k is not None:
                ranked = ranked[:top_k]
            kept_from_para = [a for a, _ in ranked]

        kept_from_para = _explicitness_filter(
            kept_from_para, original_sentence, jaccard_threshold=jaccard_explicitness
        )
        final = _dedup_preserve(pinned_originals + kept_from_para)

        if verbose:
            print(f"[PARA-k={k}] mc={mc} orig={len(pinned_originals)}, "
                  f"cands={len(candidates)}, kept_para={len(kept_from_para)}, final={len(final)}")
        return final

    out = {
        "pred_orig"     : pred_orig,
        "pred_para_list": pred_para_list,
        "k_0Preds"      : _aggregate_for_k(0),
        "k_2Preds"      : _aggregate_for_k(2),
        "k_4Preds"      : _aggregate_for_k(4),
        "k_6Preds"      : _aggregate_for_k(6),
        "k_8Preds"      : _aggregate_for_k(8),
        "k_10Preds"     : _aggregate_for_k(10),
    }
    return pd.DataFrame([out])


time: 2.28 s (started: 2026-05-03 11:25:24 +00:00)


#### Main Inference pipeline:

##### 1. Load Test Data, KG pickle object

In [ ]:
import pandas as pd

#Mount the drive:
from google.colab import drive
drive.mount('/content/drive')


# ============== 1. Load Test set ==========
test = pd.read_csv("/content/drive/MyDrive/colab_files/m-absa-arabic-test.csv")
# test=pd.read_csv("test_15")
# test= test[test['dataset'] == 'HAAD']
# test= test[test['dataset'] == 'SemEval2016']
# test = test.reset_index(drop=True)

test = process_data(test)
test

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,index,dataset,domain,sentence,aspect,category,type,source_dataset,language,paraphrases
0,0,m-absa-arabic,food,إنها تحتوي على كمية مناسبة من النكهة والرائحة.,[none],food#quality,implicit,NaN,Arabic,[1. هذه المادة تحتوي على كمية مناسبة من طعم ال...
1,1,m-absa-arabic,food,ومع ذلك، لاحظت أنه لم يعد موجودًا في المخزون و...,[none],amazon#availability,implicit,NaN,Arabic,[1. ومع ذلك، لاحظت أن المنتج لم يعد متوفراً في...
2,2,m-absa-arabic,food,أوصيت به بشدة؛ لقد أعجبني كثيرًا لدرجة أنني طل...,"[none , none]","food#recommendation , food#general",implicit,NaN,Arabic,[1. أوصيت بالمادة أو المنتج بشدة؛ لقد أعجبني ذ...
3,3,m-absa-arabic,food,أنا سعيد حقًا لأنني وجدت هذه المنتجات.,[المنتجات],food#general,explicit,NaN,Arabic,[أنا مسرور جدًا لأنني عثرت على هذه المنتجات ال...
4,4,m-absa-arabic,food,طعمه رائع ولا يوجد أي انهيار لاحقًا، فأنا أخبر...,"[none , none]","food#quality , food#general",implicit,NaN,Arabic,[1. طعم الطعام رائع ولا يوجد أي انهيار في القو...
...,...,...,...,...,...,...,...,...,...,...
2177,2177,m-absa-arabic,restaurant,في الواقع، يرغب العديد من الأشخاص بالعودة مرة ...,[none],restaurant#general,implicit,NaN,Arabic,[في الحقيقة، يرغب عدد كبير من الأفراد في العود...
2178,2178,m-absa-arabic,restaurant,– أفضل مكان مكسيكي لتناول الغداء في الحي المالي.,[مكان مكسيكي],restaurant#general,explicit,NaN,Arabic,[1. أفضل مطعم مكسيكي لتناول وجبة الغداء في منط...
2179,2179,m-absa-arabic,restaurant,أنا أحب هذا المكان!,[المكان],restaurant#general,explicit,NaN,Arabic,[أنا أحب هذا الموقع الجغرافي! \nأنا أتعشق هذه...
2180,2180,m-absa-arabic,restaurant,الجو رائع .,[الجو],ambience#general,explicit,NaN,Arabic,[الطقس في هذا اليوم جميل وممتع. \nحالة الجو ا...


time: 1.76 s (started: 2026-05-03 11:26:08 +00:00)


##### 2. Run Full pipeline:

In [ ]:
import pandas as pd
from tqdm import tqdm
import time, gc
import torch

import os, torch
# ================================================================
# STEP 1: Iterative Residual Decoding
# ================================================================
from tqdm import tqdm
tqdm.pandas()

"""
Run the full inference pipeline on a test DataFrame.
Saves intermediate results to CSV after each component.

Parameters
----------
test : pd.DataFrame
    Must contain columns ['sentence', 'aspect', 'paraphrases']
kg : KnowledgeGraph
    Knowledge graph instance
domain_embs : dict
    Precomputed domain embeddings
save_path : str
    File path to save CSV after each stage
max_rounds : int
    Max rounds for iterative residual decoding
top_k : int
    Keep top-k aspects in paraphrase step (None = keep all)
sim_threshold : float
    Similarity threshold for paraphrase filtering (None = keep all)
domain_conf_threshold : float
    Confidence threshold for domain inference in KG

Returns
-------
pd.DataFrame
    Test DataFrame with new prediction columns

Columns produced:
- explicit_iterative
- explicit_paraphrases
- implicit_kg
- combined_aspects
"""

# ================================================================
# STEP 1: Finetuned model only Extraction
# ================================================================

def run_baseline(test, save_path="test_baseline.csv"):
    print("\n[STEP 0] Baseline...")
    t0 = time.time()

    test["baseline"] = test["sentence"].progress_apply(
        lambda s: extract_aspect(s)
    )

    elapsed = time.time() - t0
    test.to_csv(save_path, index=False, encoding="utf-8")
    print(f"[SAVED] Baseline results -> {save_path}")
    print(f"[TIME] Baseline inference took {elapsed:.2f} sec")
    # --- Memory cleanup block ---
    del t0
    gc.collect()
    torch.cuda.empty_cache()
    # ----------------------------
    return test



# ================================================================
# STEP 2: Iterative Generation Extraction
# ================================================================

def run_iterative_only(test, save_path="test_iterative.csv", max_rounds=3):
    print("\n[STEP 1] Iterative Residual Decoding...")
    t0 = time.time()

    test["explicit_iterative"] = test["sentence"].progress_apply(
        lambda s: iterative_residual_decoding(s, max_rounds=max_rounds)
    )

    elapsed = time.time() - t0
    test.to_csv(save_path, index=False, encoding="utf-8")
    print(f"[SAVED] Iterative results -> {save_path}")
    print(f"[TIME] Iterative decoding took {elapsed:.2f} sec")

    # --- Memory cleanup block ---
    del t0
    gc.collect()
    torch.cuda.empty_cache()
    # ----------------------------
    return test


# ================================================================
# STEP 3: Paraphrase-Augmented Extraction
# ================================================================
def run_paraphrase_only_old(test, save_path="test_paraphrase.csv",
                        paraphrase_top_k=None, paraphrase_sim_threshold=None):
    print("\n[STEP 2] Paraphrase-Augmented Extraction...")
    t0 = time.time()

    test["cleaned_explicit_paraphrases"] = test.progress_apply(
        lambda r: extract_aspects_with_paraphrases(
            [r["sentence"]] + (r["paraphrases"] if isinstance(r["paraphrases"], list) else []),
            top_k=2,
            similarity_threshold=0.58, #0.65,
            min_consensus=1,
            include_original_always=True,
            verbose=False,
        ),
        axis=1
    )


    elapsed = time.time() - t0
    test.to_csv(save_path, index=False, encoding="utf-8")
    print(f"[SAVED] Paraphrase results -> {save_path}")
    print(f"[TIME] Paraphrase extraction took {elapsed:.2f} sec")

    # --- Memory cleanup block ---
    del t0
    gc.collect()
    torch.cuda.empty_cache()
    # ----------------------------
    return test

import time, gc, torch
from tqdm import tqdm

tqdm.pandas()

# ================================================================
# STEP 3: Paraphrase-Augmented Extraction (Edited)
# ================================================================


def run_paraphrase_only(
    test,
    input_path="/content/drive/MyDrive/colab_files/Arabic_test2.csv",
    save_path=None,
):
    """
    Runs paraphrase-aided extraction once per row (orig + 10 paraphrases),
    stores cached predictions (pred_orig, pred_para_list) and the k-sweep outputs,
    and saves to Parquet (recommended for nested lists).

    If save_path is None, it saves next to input_path with suffix '_with_para_cache.parquet'.
    """

    print("\n[STEP 2] Paraphrase-Augmented Extraction (k-sweep cached inference)...")

    # ---- Decide save path in same folder as input ----
    if save_path is None:
        base_dir = os.path.dirname(input_path)
        base_name = os.path.splitext(os.path.basename(input_path))[0]
        save_path = os.path.join(base_dir, f"{base_name}_with_para_cache.parquet")

    t0 = time.time()

    def _apply_row(r):
        paras = r["paraphrases"] if isinstance(r.get("paraphrases", None), list) else []
        sentences = [r["sentence"]] + paras  # expected 11 (1 + 10)

        df_out = extract_aspects_with_paraphrases(
            sentences,
            top_k=2,
            similarity_threshold=0.58,
            include_original_always=True,
            jaccard_explicitness=0.50,
            verbose=False,
        )

        # df_out is 1-row DataFrame
        return df_out.iloc[0].to_dict()

    results = test.progress_apply(_apply_row, axis=1)

    # Expand dict results into columns
    results_df = pd.DataFrame(list(results))

    # These are the columns we expect from extract_aspects_with_paraphrases()
    cols_to_add = [
        "pred_orig", "pred_para_list",
        "k_0Preds", "k_2Preds", "k_4Preds", "k_6Preds", "k_8Preds", "k_10Preds"
    ]

    for col in cols_to_add:
        if col in results_df.columns:
            test[col] = results_df[col].values
        else:
            # if missing for any reason, create empty column to avoid crash
            test[col] = [[] for _ in range(len(test))]

    elapsed = time.time() - t0

    # Save as Parquet (preserves nested lists)
    test.to_parquet(save_path, index=False)
    print(f"[SAVED] Paraphrase cached results -> {save_path}")
    print(f"[TIME] Paraphrase extraction took {elapsed:.2f} sec")

    # --- Memory cleanup block ---
    del t0, results, results_df
    gc.collect()
    torch.cuda.empty_cache()
    # ----------------------------

    return test


# ================================================================
# STEP 4: Knowledge Graph Inference
# ================================================================
def run_kg_only(test, kg, domain_embs, save_path="test_kg.csv", domain_conf_threshold=0.15):
    print("\n[STEP 3] Knowledge Graph Inference...")
    t0 = time.time()

    test["implicit_kg"] = test["sentence"].progress_apply(
        lambda s: infer_implicit_from_kg_best(
            s, kg, domain_embs, domain_conf_threshold=domain_conf_threshold
        )
    )



    elapsed = time.time() - t0
    test.to_csv(save_path, index=False, encoding="utf-8")
    print(f"[SAVED] KG results -> {save_path}")
    print(f"[TIME] KG inference took {elapsed:.2f} sec")

    # --- Memory cleanup block ---
    del t0
    gc.collect()
    torch.cuda.empty_cache()
    # ----------------------------
    return test


# ================================================================
# STEP 4: Combine Explicit + Implicit
# ================================================================
def run_combined_only(test, save_path="test_combined.csv"):
    print("\n[STEP 4] Combine Explicit + Implicit...")
    t0 = time.time()

    def combine_aspects(row):
        seen, combined = set(), []
        for source in [row.get("explicit_iterative", []),
                       row.get("cleaned_explicit_paraphrases", [])]:#,

            if isinstance(source, list):
                for asp in source:
                    if asp and asp.lower() not in seen:
                        seen.add(asp.lower())
                        combined.append(asp)
        return combined

    test["combined_aspects"] = test.apply(combine_aspects, axis=1)


    elapsed = time.time() - t0
    test.to_csv(save_path, index=False, encoding="utf-8")
    print(f"[SAVED] Combined results -> {save_path}")
    print(f"[TIME] Combining took {elapsed:.2f} sec")

    # --- Memory cleanup block ---
    del t0
    gc.collect()
    torch.cuda.empty_cache()
    # ----------------------------
    return test

time: 6.88 ms (started: 2026-05-03 11:26:15 +00:00)


In [ ]:
# Run each step separately
test = run_baseline(test, save_path="baseline.csv")
test = run_iterative_only(test, save_path="iterative_only.csv")


[STEP 0] Baseline...


100%|██████████| 1226/1226 [20:32<00:00,  1.01s/it]


[SAVED] Baseline results -> baseline.csv
[TIME] Baseline inference took 1232.21 sec

[STEP 1] Iterative Residual Decoding...


100%|██████████| 1226/1226 [49:12<00:00,  2.41s/it]


[SAVED] Iterative results -> iterative_only.csv
[TIME] Iterative decoding took 2952.47 sec
time: 1h 9min 46s (started: 2026-05-03 21:19:41 +00:00)


In [ ]:
test = run_paraphrase_only(test, save_path="paraphrase_only.csv")#, paraphrase_top_k=5, paraphrase_sim_threshold=0.4)


[STEP 2] Paraphrase-Augmented Extraction (k-sweep cached inference)...


100%|██████████| 2182/2182 [2:38:00<00:00,  4.34s/it]


[SAVED] Paraphrase cached results -> paraphrase_only.csv
[TIME] Paraphrase extraction took 9480.48 sec
time: 2h 38min 1s (started: 2026-05-03 12:26:04 +00:00)


##### Post-processing refinements:

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# UNIFIED post_process
# Single entry point for all four prediction columns.
# ═══════════════════════════════════════════════════════════════════════════
import re, ast
from typing import List
import pandas as pd

# Pre-compiled separator pattern
_SEP_RE = re.compile(
    r'\s*(?:\[sep\]|\[SEP\]|،|,|;|؛|\||\n|\t|/|\\\\:|\s{2,})\s*',
    re.IGNORECASE
)
_EDGE_RE = re.compile(r'^[\s\-\–\—\·\•\*\.]+|[\s\-\–\—\·\•\*\.]+$')

# ── None-value detection ────────────────────────────────────────────────────
# BUG FIX: when the model predicts multiple 'none' values they get serialised
# as a single list element: "none , none" or "none , none , none".
# ast.literal_eval then parses this as ONE string that is NOT equal to "none",
# causing it to pass through the none-filter as a phantom prediction.
# _NONE_PAT matches any comma-joined sequence of "none" tokens so all variants
# ("none", "none , none", "none , none , none", …) are correctly detected.
_NONE_PAT = re.compile(r'^(none\s*[,،]\s*)*none$', re.IGNORECASE)

def _is_none_val(t: str) -> bool:
    """True if the string is a none-only value (incl. 'none , none' variants)."""
    t = str(t).strip()
    return not t or bool(_NONE_PAT.match(t))


def _safe_parse(x) -> List[str]:
    """
    Robustly convert any raw prediction value to a flat list of strings.
    Handles: actual list, stringified list, plain string, None/NaN.
    None-only values ('none', 'none , none', etc.) are kept as-is so that
    post_process can return ['none'] for empty predictions; they are filtered
    out in prf1_for_example and evaluate_dataframe at evaluation time.
    """
    if x is None:
        return []
    if isinstance(x, list):
        items = x
    else:
        s = str(x).strip()
        try:
            if pd.isna(x):
                return []
        except Exception:
            pass
        try:
            evaled = ast.literal_eval(s)
            items = evaled if isinstance(evaled, list) else [str(evaled)]
        except Exception:
            s = s.strip("[](){}").replace("'", "").replace('"', "")
            items = _SEP_RE.split(s)
    out = []
    for t in items:
        t = _EDGE_RE.sub("", str(t).strip())
        if t:
            out.append(t)
    return out


def post_process(raw_pred, camel_obj=None, strip_adjectives=True) -> List[str]:
    """
    Unified post-processing for ALL prediction columns.

    Parameters
    ----------
    raw_pred : any
        Raw model output — list, stringified list, plain string, or NaN.
    camel_obj : CamelAdjRemover, optional
        Shared CAMeL instance. If None, get_camel() is called lazily.
    strip_adjectives : bool, default True
        Whether to apply CAMeL POS-based adjective stripping.
        Set to False for the fine-tuned baseline, which already outputs
        aspect nouns. Stripping on baseline is harmful because Arabic noun
        plurals (الموظفون, موظفين, الكماليات) share endings with adjective
        plurals and get incorrectly stripped.

    Steps
    ─────
    1. Parse input to a flat list of strings (_safe_parse).
    2. Filter out none-only values (_is_none_val).
    3. Optionally strip adjectives via CAMeL POS tags.
    4. Drop tokens that are empty or <= 2 normalized chars.
    5. Deduplicate by CAMeL-normalized key, preserving first surface form.
    6. Return ['none'] if the result is empty.
    """
    aspects = _safe_parse(raw_pred)

    # Step 2: drop none-only tokens before any further processing
    aspects = [a for a in aspects if not _is_none_val(a)]

    # Step 3: adjective stripping (only for IG/Para/Combined, NOT baseline)
    if strip_adjectives:
        try:
            camel = camel_obj or get_camel()
            stripped = []
            for a in aspects:
                s = camel.strip_adjectives(str(a)).strip()
                stripped.append(s if s else str(a).strip())
        except Exception:
            stripped = [str(a).strip() for a in aspects]
    else:
        stripped = [str(a).strip() for a in aspects]

    # Step 4: length filter
    stripped = [a for a in stripped if a and len(normalize_phrase(a)) > 2]

    # Step 5: dedup by normalized key, keep first surface form
    seen, final = set(), []
    for a in stripped:
        key = normalize_phrase(a)
        if key and key not in seen:
            seen.add(key)
            final.append(a)

    # Step 6: empty → 'none'
    return final if final else ["none"]


time: 3.05 ms (started: 2026-05-03 15:04:05 +00:00)


In [ ]:
## Case pred_para_list is a list (Not an array)
import pandas as pd
from tqdm import tqdm

tqdm.pandas()


# --------- Post-process helpers ----------
# Uses your Arabic post_process() exactly as provided.
# IMPORTANT: post_process expects either a string OR a list. It will normalize/dedup and (optionally) remove adjectives.

def _pp_list(x):
    """Post-process a prediction field that should be a list[str]."""
    if x is None:
        return []
    # sometimes parquet may store as object/np array; convert safely
    if isinstance(x, (tuple, set)):
        x = list(x)
    if isinstance(x, list):
        # keep list-of-strings as is; post_process will parse/dedup appropriately
        return post_process(x)
    # if it's a string (rare in your logged setup)
    return post_process(str(x))

def _pp_list_of_lists(x):
    """Post-process pred_para_list: list[list[str]]"""
    if x is None:
        return []
    if isinstance(x, (tuple, set)):
        x = list(x)
    if not isinstance(x, list):
        return []
    out = []
    for lst in x:
        out.append(_pp_list(lst))
    return out

# --------- 3) Apply post-processing to cached columns ----------
# These are the key cached columns you said you need:
if "pred_orig" in test.columns:
    test["pred_orig_pp"] = test["pred_orig"].progress_apply(_pp_list)

if "pred_para_list" in test.columns:
    test["pred_para_list_pp"] = test["pred_para_list"].progress_apply(_pp_list_of_lists)

# Optional but recommended: also post-process the k-sweep outputs you already saved
K_COLS = ["k_0Preds", "k_2Preds", "k_4Preds", "k_6Preds", "k_8Preds", "k_10Preds"]
for c in K_COLS:
    if c in test.columns:
        test[c + "_pp"] = test[c].progress_apply(_pp_list)

print("Done. Example:")
display(test.head(2))


100%|██████████| 2182/2182 [00:03<00:00, 716.01it/s]


Done. Example:


,index,dataset,domain,sentence,aspect,category,type,source_dataset,language,paraphrases,...,k_8Preds,k_10Preds,pred_orig_pp,pred_para_list_pp,k_0Preds_pp,k_2Preds_pp,k_4Preds_pp,k_6Preds_pp,k_8Preds_pp,k_10Preds_pp
0,0,m-absa-arabic,food,إنها تحتوي على كمية مناسبة من النكهة والرائحة.,[none],food#quality,implicit,NaN,Arabic,[1. هذه المادة تحتوي على كمية مناسبة من طعم ال...,...,[],[],[none],[[none]],[none],[none],[none],[none],[none],[none]
1,1,m-absa-arabic,food,ومع ذلك، لاحظت أنه لم يعد موجودًا في المخزون و...,[none],amazon#availability,implicit,NaN,Arabic,[1. ومع ذلك، لاحظت أن المنتج لم يعد متوفراً في...,...,[],[],[none],[[none]],[none],[none],[none],[none],[none],[none]


time: 33.4 s (started: 2026-05-03 15:04:05 +00:00)


In [ ]:
## Case pred_para_list is an array, not a list (When uploading a parquet file)

import numpy as np
from tqdm import tqdm
tqdm.pandas()

# ---------- helpers ----------
def _to_list(x):
    """Convert numpy arrays and other list-like objects into Python lists."""
    if x is None:
        return []
    if isinstance(x, np.ndarray):
        return x.tolist()
    if hasattr(x, "tolist"):   # e.g., pandas/pyarrow containers
        return x.tolist()
    if isinstance(x, (list, tuple)):
        return list(x)
    return [x]

def ensure_non_empty(lst):
    """Replace empty list with ['none']."""
    return ["none"] if not lst else lst

def post_process_list_of_aspects(aspects):
    """
    aspects: list[str] OR ndarray[str] OR string
    returns: list[str] after your Arabic post_process + empty->['none']
    """
    aspects = _to_list(aspects)
    out = post_process(aspects)   # uses your provided Arabic post_process()
    return ensure_non_empty(out)

def post_process_pred_para_list(pred_para_list):
    """
    pred_para_list: ndarray length ~10; each item is ndarray[str] or list[str]
    returns: list[list[str]] with empty inner lists -> ['none']
    """
    outer = _to_list(pred_para_list)
    out = [post_process_list_of_aspects(inner) for inner in outer]
    # If (rarely) the whole outer list is empty, keep it as 10? leave as-is:
    return out

def ensure_non_empty_nested(list_of_lists):
    """Apply empty->['none'] to each inner list (defensive)."""
    outer = _to_list(list_of_lists)
    return [ensure_non_empty(_to_list(inner)) for inner in outer]

# ---------- 1) build pp columns ----------
if "pred_orig" in test.columns:
    test["pred_orig_pp"] = test["pred_orig"].progress_apply(post_process_list_of_aspects)

if "pred_para_list" in test.columns:
    test["pred_para_list_pp"] = test["pred_para_list"].progress_apply(post_process_pred_para_list)

# ---------- 2) also fix existing k-sweep columns (if present) ----------
K_COLS = ["k_0Preds", "k_2Preds", "k_4Preds", "k_6Preds", "k_8Preds", "k_10Preds"]
for col in K_COLS:
    if col in test.columns:
        test[col] = test[col].progress_apply(post_process_list_of_aspects)

# (Optional) If you already had *_pp versions, ensure they also follow empty->['none']
for col in ["pred_orig_pp", "pred_para_list_pp"] + [c + "_pp" for c in K_COLS]:
    if col in test.columns:
        if col == "pred_para_list_pp":
            test[col] = test[col].apply(ensure_non_empty_nested)
        else:
            test[col] = test[col].apply(ensure_non_empty)

# # ---------- sanity check ----------
# i = 4
# if "pred_para_list_pp" in test.columns:
#     # print("Example pred_orig_pp:", test.loc[i, "pred_orig_pp"])
#     print("Example pred_para_list_pp[0]:", test.loc[i, "pred_para_list_pp"][0])
#     print("Example pred_para_list_pp lens:", len(test.loc[i, "pred_para_list_pp"]))


100%|██████████| 2182/2182 [00:03<00:00, 693.45it/s]

time: 33.8 s (started: 2026-05-03 15:04:39 +00:00)


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Apply unified post_process to the four prediction columns
# ═══════════════════════════════════════════════════════════════════════════
from tqdm import tqdm
tqdm.pandas()

# 1) Baseline — NO adjective stripping.
#    The fine-tuned model already outputs aspect nouns; stripping incorrectly
#    removes valid aspects with Arabic plural endings (الموظفون, موظفين, الكماليات).
test["pp_baseline"] = test["baseline"].progress_apply(
    lambda x: post_process(x, strip_adjectives=False)
)

# 2) IG component — WITH adjective stripping (IG can output sentiment phrases)
test["pp_ig"] = test["explicit_iterative"].progress_apply(
    lambda x: post_process(x, strip_adjectives=True)
)

# 3) Paraphrase — k_2Preds column — WITH stripping
test["pp_para"] = test["k_2Preds"].progress_apply(
    lambda x: post_process(x, strip_adjectives=True)
)

# 4) Combined = union of IG + Para, deduped by normalized key — WITH stripping
def combine_ig_para(row):
    ig   = post_process(row["explicit_iterative"], strip_adjectives=True)
    para = post_process(row["k_2Preds"],           strip_adjectives=True)
    seen, merged = set(), []
    for a in ig + para:
        key = normalize_phrase(a)
        if key and key != "none" and key not in seen:
            seen.add(key)
            merged.append(a)
    return merged if merged else ["none"]

test["pp_combined"] = test.progress_apply(combine_ig_para, axis=1)

# 5) Gold — consistent list format with none-unification
def clean_gold(x):
    items = _safe_parse(x)
    items = [i for i in items if str(i).strip()]
    return items if items else ["none"]

test["aspect"] = test["aspect"].apply(clean_gold)

print("Post-processing done.")
print(f"  pp_baseline : {test['pp_baseline'].apply(len).mean():.2f} avg aspects/row")
print(f"  pp_ig       : {test['pp_ig'].apply(len).mean():.2f} avg aspects/row")
print(f"  pp_para     : {test['pp_para'].apply(len).mean():.2f} avg aspects/row")
print(f"  pp_combined : {test['pp_combined'].apply(len).mean():.2f} avg aspects/row")


100%|██████████| 2182/2182 [00:05<00:00, 403.58it/s]

Post-processing done.
  pp_baseline : 1.00 avg aspects/row
  pp_ig       : 1.07 avg aspects/row
  pp_para     : 1.10 avg aspects/row
  pp_combined : 1.24 avg aspects/row
time: 10.9 s (started: 2026-05-03 15:05:13 +00:00)


In [ ]:
test.to_csv("m-absa_predictions_v4_pp.csv")
from google.colab import files
files.download('m-absa_predictions_v4_pp.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

time: 196 ms (started: 2026-05-03 15:05:23 +00:00)


## Evaluation:

#### New Evaluation function:

Evaluation pipeline:

1. post-process test['preidction'] (parse → remove adjectives → normalize → dedup)

2. parse gold aspects

3. compute precision/recall/F1 per row and overall (micro & macro)

In [ ]:
# --- similarity, best_alignment, prf1_for_example ---

def jaccard(a: str, b: str) -> float:
    A = set(a.split()); B = set(b.split())
    if not A and not B: return 1.0
    if not A or not B:  return 0.0
    return len(A & B) / len(A | B)

def similarity(gold: str, pred: str,
               allow_substring: bool = True,
               use_jaccard: bool = True) -> float:
    g = normalize_phrase(gold)
    p = normalize_phrase(pred)
    if not g or not p: return 0.0
    if g == p: return 1.0
    if allow_substring and (g in p or p in g): return 1.0
    return jaccard(g, p) if use_jaccard else 0.0

def best_alignment(gold, pred, thresh=0.6,
                   allow_substring: bool = True,
                   use_jaccard: bool = True):
    pairs = []
    for i, g in enumerate(gold):
        for j, p in enumerate(pred):
            s = similarity(g, p, allow_substring=allow_substring, use_jaccard=use_jaccard)
            if s >= thresh:
                pairs.append((i, j, s))
    pairs.sort(key=lambda x: x[2], reverse=True)
    matched_g, matched_p, matches = set(), set(), []
    for gi, pj, s in pairs:
        if gi not in matched_g and pj not in matched_p:
            matched_g.add(gi); matched_p.add(pj); matches.append((gi, pj, s))
    return len(matches), matches

def prf1_for_example(gold_terms, pred_terms,
                     thresh: float = 0.6,
                     allow_substring: bool = True,
                     use_jaccard: bool = True):
    gold = [t for t in map(str, gold_terms) if not _is_none_val(t)]
    pred = [t for t in map(str, pred_terms) if not _is_none_val(t)]

    if not gold and not pred:
        return {"tp": 0, "fp": 0, "fn": 0,
                "precision": 1.0, "recall": 1.0, "f1": 1.0, "exact": 1.0}

    tp, _ = best_alignment(gold, pred, thresh=thresh,
                           allow_substring=allow_substring, use_jaccard=use_jaccard)
    fp = max(0, len(pred) - tp)
    fn = max(0, len(gold) - tp)

    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec  = tp / (tp + fn) if (tp + fn) else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0

    gset = {normalize_phrase(x) for x in gold}
    pset = {normalize_phrase(x) for x in pred}
    exact = 1.0 if gset == pset else 0.0

    return {"tp": tp, "fp": fp, "fn": fn,
            "precision": prec, "recall": rec, "f1": f1, "exact": exact}


time: 2.52 ms (started: 2026-05-03 15:05:24 +00:00)


In [ ]:
def evaluate_dataframe(
    df: pd.DataFrame,
    pred_col: str = "baseline",
    gold_col: str = "aspectTerms",
    jaccard_threshold: float = 0.6,
    allow_substring: bool = True,
    use_jaccard: bool = True,
    camel_obj=None,
    normalize_gold: bool = True,
    already_processed: bool = False,
):
    """
    Evaluate an aspect extraction column against a gold column.

    Parameters
    ----------
    already_processed : bool, default False
        Set True when pred_col already contains post-processed lists
        (e.g. pp_baseline, pp_ig, pp_para, pp_combined).
        Skips the post_process() call to avoid double-processing and
        incorrect adjective stripping on already-clean predictions.
        When False (default), runs post_process() on the raw pred_col.
    """
    df = df.copy()
    camel_obj = get_camel()

    # 1) predictions → cleaned lists
    if already_processed:
        # Column is already a list (or stringified list) — just parse + none-filter
        df["pred_list_clean"] = df[pred_col].apply(
            lambda s: [t for t in _safe_parse(s) if not _is_none_val(t)]
        )
    else:
        # Raw column — run full post_process pipeline
        df["pred_list_clean"] = df[pred_col].apply(
            lambda s: [t for t in post_process(s, camel_obj=camel_obj)
                       if not _is_none_val(t)]
        )

    # 2) gold → lists, filtered of none values
    def to_list(x):
        items = x if isinstance(x, list) else parse_aspect_string(x)
        return [t for t in items if not _is_none_val(str(t))]
    df["gold_list"] = df[gold_col].apply(to_list)

    # 3) optional: normalize/dedup gold for fair exact accuracy & support
    if normalize_gold:
        df["gold_list"] = df["gold_list"].apply(normalize_and_dedup_list)

    # 4) per-row metrics
    per = df.apply(
        lambda r: prf1_for_example(
            r["gold_list"], r["pred_list_clean"],
            thresh=jaccard_threshold,
            use_jaccard=use_jaccard
        ),
        axis=1
    )
    df["precision"] = per.apply(lambda m: m["precision"])
    df["recall"]    = per.apply(lambda m: m["recall"])
    df["f1"]        = per.apply(lambda m: m["f1"])
    df["tp"]        = per.apply(lambda m: m["tp"])
    df["fp"]        = per.apply(lambda m: m["fp"])
    df["fn"]        = per.apply(lambda m: m["fn"])
    df["exact_row"] = per.apply(lambda m: m["exact"])
    df["support"]   = df["gold_list"].apply(len)

    # 5) aggregates
    tot_tp, tot_fp, tot_fn = int(df["tp"].sum()), int(df["fp"].sum()), int(df["fn"].sum())
    micro_p = tot_tp / (tot_tp + tot_fp) if (tot_tp + tot_fp) else 0.0
    micro_r = tot_tp / (tot_tp + tot_fn) if (tot_tp + tot_fn) else 0.0
    micro_f = 2 * micro_p * micro_r / (micro_p + micro_r) if (micro_p + micro_r) else 0.0
    micro_acc_jaccard = tot_tp / (tot_tp + tot_fp + tot_fn) if (tot_tp + tot_fp + tot_fn) else 0.0

    nrows = len(df)
    macro_p = df["precision"].mean() if nrows else 0.0
    macro_r = df["recall"].mean()    if nrows else 0.0
    macro_f = df["f1"].mean()        if nrows else 0.0

    total_support = df["support"].sum()
    if total_support > 0:
        weighted_p = (df["precision"] * df["support"]).sum() / total_support
        weighted_r = (df["recall"]    * df["support"]).sum() / total_support
        weighted_f = (df["f1"]        * df["support"]).sum() / total_support
    else:
        weighted_p = weighted_r = weighted_f = 0.0

    accuracy_exact = df["exact_row"].mean() if nrows else 0.0

    return {
        "micro": {
            "precision": round(float(micro_p), 4),
            "recall":    round(float(micro_r), 4),
            "f1":        round(float(micro_f), 4),
            "accuracy_jaccard": round(float(micro_acc_jaccard), 4),
            "tp": tot_tp, "fp": tot_fp, "fn": tot_fn,
        },
        "macro": {
            "precision": round(float(macro_p), 4),
            "recall":    round(float(macro_r), 4),
            "f1":        round(float(macro_f), 4),
        },
        "weighted_macro": {
            "precision": round(float(weighted_p), 4),
            "recall":    round(float(weighted_r), 4),
            "f1":        round(float(weighted_f), 4),
        },
        "accuracy_exact": round(float(accuracy_exact), 4),
        "df": df,
    }


time: 3.95 ms (started: 2026-05-03 15:05:24 +00:00)


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Main Evaluation — three views, all four post-processed columns
# ═══════════════════════════════════════════════════════════════════════════

_EVAL_COLS = [
    ("Baseline",           "pp_baseline"),
    ("IG",                 "pp_ig"),
    ("IG + Para (k=2)",    "pp_para"),
    ("Combined (IG+Para)", "pp_combined"),
]

# ── View subsets ─────────────────────────────────────────────────────────────
# View 1: All rows (official — matches published evaluation protocol)
# View 2: Aspect extraction only — rows where gold has at least one real aspect.
#         Excludes implicit examples labelled gold=["none"] which generate trivial
#         none==none matches that inflate the baseline disproportionately.
# View 3: Explicit only — rows where type == "explicit" (clearest for thesis)

def _has_real_aspect(x):
    """True if at least one gold label is a real aspect (not a none variant)."""
    return any(not _is_none_val(str(t)) for t in _safe_parse(x))

_views = [
    ("All rows (official)",                    test),
    ("Aspect extraction only (excl. none-gold)", test[test["aspect"].apply(_has_real_aspect)]),
    ("Explicit only (type tag)",               test[test["type"] == "explicit"]),
]

for view_name, subset in _views:
    print(f"\n=== {view_name}  (n={len(subset)}) ===")
    for label, col in _EVAL_COLS:
        if col not in subset.columns:
            print(f"  [skip] {label}: column '{col}' not found")
            continue
        result = evaluate_dataframe(
            df=subset,
            pred_col=col,
            gold_col="aspect",
            jaccard_threshold=0.6,
            use_jaccard=True,
            normalize_gold=True,
            already_processed=True,   # pp_* columns are already post-processed
        )
        m = result["micro"]
        print(f"  {label:<22}  P={m['precision']:.4f}  R={m['recall']:.4f}  "
              f"F1={m['f1']:.4f}  TP={m['tp']:5d}  FP={m['fp']:5d}  FN={m['fn']:5d}")



=== All rows (official)  (n=2182) ===
  Baseline                P=0.6758  R=0.5568  F1=0.6106  TP=  813  FP=  390  FN=  647
  IG                      P=0.7630  R=0.7212  F1=0.7415  TP= 1053  FP=  327  FN=  407
  IG + Para (k=2)         P=0.7243  R=0.7342  F1=0.7293  TP= 1072  FP=  408  FN=  388
  Combined (IG+Para)      P=0.6138  R=0.7959  F1=0.6931  TP= 1162  FP=  731  FN=  298

=== Aspect extraction only (excl. none-gold)  (n=1460) ===
  Baseline                P=0.7144  R=0.5568  F1=0.6259  TP=  813  FP=  325  FN=  647
  IG                      P=0.8106  R=0.7212  F1=0.7633  TP= 1053  FP=  246  FN=  407
  IG + Para (k=2)         P=0.7723  R=0.7342  F1=0.7528  TP= 1072  FP=  316  FN=  388
  Combined (IG+Para)      P=0.6584  R=0.7959  F1=0.7206  TP= 1162  FP=  603  FN=  298

=== Explicit only (type tag)  (n=1356) ===
  Baseline                P=0.7479  R=0.5841  F1=0.6559  TP=  792  FP=  267  FN=  564
  IG                      P=0.8208  R=0.7330  F1=0.7744  TP=  994  FP=  217  FN=  3